<a href="https://colab.research.google.com/github/JoseAlberto88/Hugging-Face-LLM-Course/blob/main/06_The_Tokenizer_Library.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Introduction

IN chapter 3, we looked at how to fine-tune a model on a given task. When we do that, we use the same tokenizer that the model was pretrained with -but what do we do when we want to train a model from scratch? In this cases, using a tokenizer that was pretrained on a corpus from another domain or language is typically suboptimal. For example, a tokenizer that's trained on an English corpus will perform poorly of Japanese texts because the use of spaces and punctuation is very different in the two languages.

In this chapter, you will learn how to train a brand new tokenizer on a corpus of texts, so it can then be used to pretrain a language model. This will all be done with the help of the `Tokenizers` https://github.com/huggingface/tokenizers library, which provides the "fast" tokenizers in the Transformers: https://github.com/huggingface/transformers library. We'll take a close look at the features that this library provides, and explore how the fast tokenizers differ from the "slow" versions.

Topics that will cover include:

  - How to train a new tokenizer similar to the one used by a given checkpoint on a new corpus of texts
  - The special features of fast tokenizers
  - The differences between the three main subword tokenization algorithms used in NLP today.
  - How to build a tokenizer from scratch with the tokenizer library and train it on some data

The techniques introduced in this chapter will prepare you for the section in Chapter 7 where we look at creating a language model for Python source code. Let's start by looking at what it means to "train" a tokenizer in the first place.

## Training a new tokenizer from a old one

If a language model is not available in the language you are interested in, or if your corpus is very different from the one your language model was trained on, you will most likely want to retrain the model from scratch using a tokenizer adapted to your data. That will require training a new tokenizer on your dataset. But what exactly does that mean? When we first looked at tokenizers in Chapter 2, we saw that most Transformers models use a `subword tokenization algorithm`. To identify which subwords are of interest and occurs most frequently in the corpus at hand, the tokenizer needs to take a hard look at all the texts in the corpus -a process we call `training`. The exact rules that govern this training depend of the type of tokenizer used, and we'll go over the three main algorithms later in this chapter.

### Assembling a corpus

There's a very simple API in Transformers that you can use to train a new tokenizer with the same characteristics as an existing one: `AutoTokenizer.train_new_from_iterator()`. To see this in action, let's say we want to train GPT-2 from scratch, but in a language other than English. Our first task will be to gather lots of data in that language in a training corpus. To provide examples everyone will be able to understand, we won't use a language like Russian or Chinese here, but rather a specialized English language: Python code.

The Datasets:https://github.com/huggingface/datasets library can help us assemble a corpus of Python source code. We'll use the usual `load_dataset()` function to download and cache the *CodeSearchNet* dataset. This dataset was created for the CodeSearchNet Challenge: https://forge.coreweave.com/wandb/github/CodeSearchNet and contains millions of functions from open source libraries on GitHub in several programming languages. Here, we will load the Python part of this dataset:


In [1]:
from datasets import load_dataset

# This can take a few minutes to load, so grab a coffee or tea while you wait!
raw_datasets = load_dataset("code-search-net/code_search_net", "python")

README.md:   0%|          | 0.00/14.1k [00:00<?, ?B/s]

python/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  522MB            

python/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

python/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 28.7MB            

python/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

python/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 30.7MB            

python/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/412178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/22176 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/23107 [00:00<?, ? examples/s]

We can have a look at the training split to see which columns we have access to:

In [2]:
raw_datasets["train"]

Dataset({
    features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'],
    num_rows: 412178
})

We can see the datasets separates docstrings from code and suggests a tokenization of both. Here, we'll just use the `whole_func_string` column to train our tokenizer. We can look at an example of one these functions by indexing into the `train` split:

In [3]:
print(raw_datasets["train"][123456]["whole_func_string"])

def oauth_token_create(self, data, **kwargs):
        "https://developer.zendesk.com/rest_api/docs/core/oauth_tokens#create-token"
        api_path = "/api/v2/oauth/tokens.json"
        return self.call(api_path, method="POST", data=data, **kwargs)


The first thing we need to do id transform the dataset into an `iterator` of lists of texts - for instance, a list of list of texts. Using lists of texts will enable our tokenizer to go faster (training on batches of texts instead of processing individuals texts one by one), and it should be an iterator if we want to avoid having everything in memory at once. If your corpus is huge, you will want to take advantage of the fact that Datasets does not load everything into RAM but stores the elements of the dataset on disk.

Doing the following would create a list of lists of 1,000 texts each, but would load everything in memory:


In [4]:
# Don't uncomment the following line unless your dataset is small!

# training_corpus = [raw_datasets["train"][i: i+1000]["whole_func_string"] for i in range(0, len(raw_datasets["train"]), 1000)]

Using Python generator, we can avoid Python loading anything into memory until it's actually necessary. To create such as generator, you just need to replace the brackets with parentheses:

In [5]:
training_corus = (
    raw_datasets["train"][i : i + 1000]["whole_func_string"]
    for i in range(0, len(raw_datasets["train"]), 1000)
)

This line of code doesn't fetch any elements of the dataset; it just creates an object you can use in a Pyhton `for` loop. The texts will only be loaded when you need them (that is, when you're at the step of the `for` loop that requires them), and only 1,000 texts at a time will be loaded. This way you won't exhaust all your memory even if you are processing a huge dataset.

The problem with a generator object is that it can be only be used once. So, Instead of this giving us the first 10 digits twice:

In [6]:
gen = (i for i in range(10))
print(list(gen))
print(list(gen))

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[]


We get them once and then an empty list:

That's why we define a function that returns a generator instead:

In [7]:
def get_training_corpus():
  return (
      raw_datasets["train"][i : i + 1000]["whole_func_string"]
      for i in range(0, len(raw_datasets["train"]), 1000)
  )
training_corpus = get_training_corpus()

You can also define your generator inside a `for` loop by using the `yield` statement:

In [8]:
def get_training_corpus():
  dataset = raw_datasets["train"]
  for start_idx in range(0, len(dataset), 1000):
    samples = dataset[start_idx : start_idx + 1000]
    yield samples["whole_func_string"]


which will produce the exact same generator as before, but allows you to use complex logic than you can in a list comprehension.

### Training a new tokenizer

Now that we have a corpus in the form of an iterator of batches of texts, we are ready to train a new tokenizer. To do this, we first need to load the tokenizer we want to pair with our model (here, GPT-2):

In [9]:
from transformers import AutoTokenizer

old_tokenizer = AutoTokenizer.from_pretrained("gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Even though we are going to train a new tokenizer, it's a good idea to do this to avoid starting entirely from scratch. This way, we won't have to specify anything about the tokenization algorithm or the special tokens we want to use; our new tokenizer will be exactly the same as GPT-2, and the only thing that will change is the vocabulary, which will be determined by the training on our corpus.

First let's have a look at how this tokenizer would treat an example function:

In [10]:
example = '''def add_numbers(a,b):
          """Add the two numbers `a` and `b`."""
          return a + b'''

tokens = old_tokenizer.tokenize(example)
tokens

['def',
 'Ġadd',
 '_',
 'n',
 'umbers',
 '(',
 'a',
 ',',
 'b',
 '):',
 'Ċ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ"""',
 'Add',
 'Ġthe',
 'Ġtwo',
 'Ġnumbers',
 'Ġ`',
 'a',
 '`',
 'Ġand',
 'Ġ`',
 'b',
 '`',
 '."',
 '""',
 'Ċ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġreturn',
 'Ġa',
 'Ġ+',
 'Ġb']

This tokenizer has a few special symbolsm like `Ġ` and `Ċ`, which denote spaces and newlines, respectively. As we can see, this is not too efficient: the tokenizer returns individual tokens for each space, when it could group together indentation levels (since having sets of four or eight spaces is going to be very common in code). It also split the function name a bit weirdly, not being used to seeing words with the `_` character.

Let's train a new tokenizer and see if it solves those issues. For this, we'll use the method `train_new_from_iterator()`:

In [11]:
tokenizer = old_tokenizer.train_new_from_iterator(training_corpus, 52,000)

This command might take a bit of time if your corpus is very large, but for this dataset of 1.6 GB of texts it's blazing fast (1 minute 16 seconds on an AMD Ryzen 9 3900X CPU with 12 cores).

Note that ``AutoTokenizer.train_new_from_iterator()` only works if the tokenizer you are using is a "fast" tokenizer. As you'll see in the next section, the Transformers library contains two types of tokenizers: some are written purely in Python and others (the fast ones) are backed by the Tokenizers library, which is written in the Rust programming language. Python is the language most often used for data science and deep learning applications, but when anything needs to be parallelized to be fast, it has to be written in another language. For instance, the matrix multiplications that are at the core of the model computation are written in CUDA, an optimized C library for GPUs.

Training a brand new tokenizer in pure Pyhton would be excruciatingly slow, which is why we developed the Tokenizers library. Note that just as you didn't have to learn the CUDA language to be able to execute your model on a batch of inputs on a GPU, you won't need to learn Rust to use a fast tokenizer. The Tokenizers library provides Python bindings for many methods that internally call some piece of code in Rust; for example, to parallelize the training of your new tokenizer or, as we saw in chapter 3, the tokenization of a batch of inputs.

Most of the Transformers models have a fast tokenizer available (there are some exceptions that you can check here: https://huggingface.co/docs/transformers/index#supported-frameworks), and the `AutoTokenizer` API always selects the fast tokenizer for you if it's available. In the next section we'll take a look at some of the other special features fast tokenizers have, which will be really useful for tasks like token classification and question answering. Before diving into that, however, let's try our brand new tokenizer on the previous example:

In [12]:
tokens = tokenizer.tokenize(example)
tokens

['d',
 'e',
 'f',
 'Ġ',
 'a',
 'd',
 'd',
 '_',
 'n',
 'u',
 'm',
 'b',
 'e',
 'r',
 's',
 '(',
 'a',
 ',',
 'b',
 ')',
 ':',
 'Ċ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 '"',
 '"',
 '"',
 'A',
 'd',
 'd',
 'Ġ',
 't',
 'h',
 'e',
 'Ġ',
 't',
 'w',
 'o',
 'Ġ',
 'n',
 'u',
 'm',
 'b',
 'e',
 'r',
 's',
 'Ġ',
 '`',
 'a',
 '`',
 'Ġ',
 'a',
 'n',
 'd',
 'Ġ',
 '`',
 'b',
 '`',
 '.',
 '"',
 '"',
 '"',
 'Ċ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'r',
 'e',
 't',
 'u',
 'r',
 'n',
 'Ġ',
 'a',
 'Ġ',
 '+',
 'Ġ',
 'b']

Here we again see the special symbols `Ġ` and `Ċ` that denote spaces and newlines, but we can also see that our tokenizer learned some tokens that are highly specific to a corpus of Python functions: for example, there is a `ĊĠĠĠ` token that represents an indentation, and a `Ġ"""` token that represents the three quotes that start a docstring. The tokenizer also correctly split the function name on `_`. This is quite a compact representation; comparatively, using the plain English tokenizer on the same example will give us a longer sentence:

In [13]:
print(len(tokens))
print(len(old_tokenizer.tokenize(example)))

93
48


Let's look at another example:

In [14]:
example = """class LinearLayer():
    def __init__(self, input_size, output_size):
        self.weight = torch.randn(input_size, output_size)
        self.bias = torch.zeros(output_size)

    def __call__(self, x):
        return x @ self.weights + self.bias
    """
tokenizer.tokenize(example)

['c',
 'l',
 'a',
 's',
 's',
 'Ġ',
 'L',
 'i',
 'n',
 'e',
 'a',
 'r',
 'L',
 'a',
 'y',
 'e',
 'r',
 '(',
 ')',
 ':',
 'Ċ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'd',
 'e',
 'f',
 'Ġ',
 '_',
 '_',
 'i',
 'n',
 'i',
 't',
 '_',
 '_',
 '(',
 's',
 'e',
 'l',
 'f',
 ',',
 'Ġ',
 'i',
 'n',
 'p',
 'u',
 't',
 '_',
 's',
 'i',
 'z',
 'e',
 ',',
 'Ġ',
 'o',
 'u',
 't',
 'p',
 'u',
 't',
 '_',
 's',
 'i',
 'z',
 'e',
 ')',
 ':',
 'Ċ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 's',
 'e',
 'l',
 'f',
 '.',
 'w',
 'e',
 'i',
 'g',
 'h',
 't',
 'Ġ',
 '=',
 'Ġ',
 't',
 'o',
 'r',
 'c',
 'h',
 '.',
 'r',
 'a',
 'n',
 'd',
 'n',
 '(',
 'i',
 'n',
 'p',
 'u',
 't',
 '_',
 's',
 'i',
 'z',
 'e',
 ',',
 'Ġ',
 'o',
 'u',
 't',
 'p',
 'u',
 't',
 '_',
 's',
 'i',
 'z',
 'e',
 ')',
 'Ċ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ',
 's',
 'e',
 'l',
 'f',
 '.',
 'b',
 'i',
 'a',
 's',
 'Ġ',
 '=',
 'Ġ',
 't',
 'o',
 'r',
 'c',
 'h',
 '.',
 'z',
 'e',
 'r',
 'o',
 's',
 '(',
 'o',
 'u',
 't',
 'p',
 'u',
 't'

In addition to the token corresponding to an indentation, here we can also see a token for a double indentation: `ĊĠĠĠĠĠĠĠ`. The special Python words like `class`, `init`, `call`, `self`, and `return` are each tokenized as one token, and we can see that as well as splitting on `_` and `.` the tokenizer correctly splits even camel-cased names: `LinearLayer` is tokenized as `["ĠLinear", "Layer"]`.

### Saving the tokenizer

To make sure we can use it later, we need to save our new tokenizer. Like for models, this is done with the `save_pretrained()` method:

In [14]:
tokenizer.save_pretrained("code-search-net-tokenizer")

This will create a new folder named `code-search-net-tokenizer`, which will contain all the files the tokenizer needs to be reloaded. If you want to share this tokenizer with your colleagues and friends, you can upload it to the Hub by logging into your account. If you're working in a notebook, there's a convenience function to help you with this:

In [15]:
from huggingface_hub import notebook_login

notebook_login()

This will display a widget where you can enter your Hugging Face login credentials. If you aren't working in a notebook, just type the following line in your terminal:

```python
huggingface-cli login
```

Once you've logged in, you can push your tokenizer by executing the following command:

In [16]:
tokenizer.push_to_hub("code-search-net-tokenizer")

CommitInfo(commit_url='https://huggingface.co/JoseA1988/code-search-net-tokenizer/commit/8f53bbf754b3ff2d4356879963d4244e991dc02b', commit_message='Upload tokenizer', commit_description='', oid='8f53bbf754b3ff2d4356879963d4244e991dc02b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/JoseA1988/code-search-net-tokenizer', endpoint='https://huggingface.co', repo_type='model', repo_id='JoseA1988/code-search-net-tokenizer'), pr_revision=None, pr_num=None)

This will create a new repository in your namespace with the name `code-search-net-tokenizer`, containing the tokenizer file. You can then load the tokenizer from anywhere with the `from_pretrained()` method:

```python
# Replace "huggingface-course" below with your actual namespace to use your own tokenizer
tokenizer = AutoTokenizer.from_pretrained("huggingface-course/code-search-net-tokenizer")
```

You're now all set for training a language model from scratch and fine-tining it on your task at hand! We'll get to that in Chapter 7, but first, in the rest of this chapter we'll take a closer look at fast tokenizers and explore in detail what actually happens when we call the method `train_new_from_iterator()`.

## Fast tokenizer's special powers

In this section we will take a closer look at the capabilities of the tokenizers in Transformers. Up to now we have only used them to tokenize inputs or decode IDs back into text, but tokenizers -especially those backed by the Tokenizers library - can do a lot more. To illustrate these additional features, we will explore how to reproduce the results of the `token-classification` (that we called `ner`) and `question-answering` pipelines that we first encountered in Chapter 1.

In the following discussion, we will often make the distinction between "slow" and "fast" tokenizers. low tokenizers are those written in Python inside the Transformers library, while the fast versions are the ones provided by Tokenizers, which are written in Rust. If you remember the table from Chapter 5 that reported how long it took a fast and a slow tokenizer to tokenize the Drug Review Dataset, you should have and idea of why we call them fast and slow:

| | Fast tokenizer | Slow tokenizer |
|---|---|---|
| batched=True | 10.8s | 4min41s |
| batched=False | 59.2s | 5min3s |



### Batch encoding

The output of a tokenizer isn't a simple Python dictionary; what we get is actually a special `BatchEncoding` object. It's a subclass of a dictionary (which is why we were able to index into that result without any problem before), but with additional methods that are mostly used by fast tokenizers.

Besides their parallelization capabilities, the key functionality of fast tokenizers is that they always keep track if the original span of texts the final tokens come from -a feature we call `offset mapping`. This in turn unlocks features like mapping each word to the tokens it generated or mapping each character of the original text to the token it's inside, and vice versa.

Let's take a look at an example:

In [17]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
example = "My name is sylvain and I work at Hugging Face in Brooklyn."
encoding = tokenizer(example)
print(type(encoding))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

<class 'transformers.tokenization_utils_base.BatchEncoding'>


As mentioned previously, we get a `BatchEncoding` object in the tokenizer's output:

```python
<class 'transformers.tokenization_utils_base.BatchEncoding'>

```

Since the `AutoTokenizer` class picks a fast tokenizer by default, we can use the additional methods this   `BatchEncoding` object provides. We have two ways to check if our tokenizer is a fast or a slow one. We can either check the attribute `is_fast` of the `tokenizer`:

In [18]:
tokenizer.is_fast

True

or check the same attribute of our `encoding`:

In [19]:
encoding.is_fast

True

Let's see what a fast tokenizer enables us to do. First, we can access the tokens without having to convert the IDs back to tokens:

In [20]:
encoding.tokens()

['[CLS]',
 'My',
 'name',
 'is',
 's',
 '##yl',
 '##va',
 '##in',
 'and',
 'I',
 'work',
 'at',
 'Hu',
 '##gging',
 'Face',
 'in',
 'Brooklyn',
 '.',
 '[SEP]']

In this case the token at index 5 is `##yl`, which is part of the word “Sylvain” in the original sentence. We can also use the `word_ids()` method to get the index of the word each token comes from:

In [21]:
encoding.word_ids()

[None, 0, 1, 2, 3, 3, 3, 3, 4, 5, 6, 7, 8, 8, 9, 10, 11, 12, None]

We can see that the tokenizer's special tokens `[CLS]` and `[SEP]` are mapped to `None`, and then each token is mapped to the word it originates from. This is especially useful to determine if a token is at the start of a word or if two tokens are in the same word. We could rely in the `##` prefix for that, but it only works for BERT-like tokenizers; this method works for any type of tokenizer as long as it's a fast one. In the next chapter, we'll see how we can use this capability to apply the labels we have for each word proprely to the tokens in tasks like named entiry recognition (NER) and part-of-speech (POS) tagging. We can also use it to mask all the tokens coming from the same word in masked language modelig (a technique called `whole word masking`).

Similarly, there is a `sentence_ids()` method that we can use to map a token to the sentence it came from (though in this case, the `token_type_ids` returned by the tokenizer can give us the same information).

Lastly, we can map any word or token to characters in the original text, and vice versa, via the `word_to_chars()` or `token_to_chars()` and `char_to_word()` or `char_to_token()` methods. For instance, the `word_ids()` method told us that `##yl` is part of the word at index 3, but which word is it in the sentence? We can find out like this:

In [23]:
start, end = encoding.word_to_chars(3)
example[start:end]

'sylvain'

As we mentioned previously, this is all powered by the fact the fast tokenizer keeps track of the span of text each token come from in a list of `offsets`. To illustrate their use, next we'll show you how to replicate the results of the `token-classification` pipeline manually.

### Inside the token-classification pipeline